In [8]:
# LITHIUM

from mp_api.client import MPRester
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tabulate import tabulate

API_KEY = "4IeHY5jVcrgiKXNuAo6Jgs7yC0Z3hsli"
with MPRester(API_KEY) as mpr:
    # Query for Li-based cathode materials with high energy density
    docs = mpr.materials.insertion_electrodes.search(
    working_ion="Li", average_voltage = (0.0, 6.5), stability_charge=(0.0, 0.5), max_delta_volume=(0,1),
        # 0.1 > stability_charge (meV) starts to be unstable
    fields=[
        "battery_id", "formula_discharge", "average_voltage"
        , "energy_grav", "energy_vol", "capacity_grav", "capacity_vol", "stability_charge", "fracA_charge", "max_delta_volume", "stability_discharge", "fracA_discharge"
    ]
)



df = pd.DataFrame([doc.dict() for doc in docs])
print(df.shape)





Retrieving InsertionElectrodeDoc documents: 100%|██████████| 2647/2647 [00:00<00:00, 3895.37it/s]

(2647, 13)


In [24]:
# Choose your 8 properties and whether to minimize or maximize each
objectives = {
    "average_voltage":    "max",
    "capacity_grav":      "max",
    "energy_grav":        "max",
    "fracA_charge":        "min",   # higher = more stable
    "fracA_discharge":     "max",
    "stability_charge":   "min",
    "stability_discharge": "min",
    "max_delta_volume":   "min",   # lower volume change = better
}

cols = list(objectives.keys())
df_clean = df[cols + "battery_id", "formula_discharge"].dropna().copy()

df_ids = df_clean[["battery_id", "formula_discharge"]].copy()


# Convert everything to minimization (negate maximization objectives)
df_min = df_clean.copy()
for col, direction in objectives.items():
    if direction == "max":
        df_min[col] = -df_clean[col]

costs = df_min.values  # shape: (n_materials, 8)

KeyError: "None of [Index(['battery_id', 'formula_discharge'], dtype='object')] are in the [columns]"

In [13]:
def pareto_front_fast(costs):
    """
    Find non-dominated solutions. All objectives assumed minimization.
    Returns boolean mask of Pareto-optimal rows.
    """
    costs = np.array(costs)
    n = len(costs)
    is_pareto = np.ones(n, dtype=bool)

    for i in range(n):
        if is_pareto[i]:
            # Find all solutions that dominate solution i
            dominated = (
                np.all(costs[is_pareto] <= costs[i], axis=1) &
                np.any(costs[is_pareto] <  costs[i], axis=1)
            )
            dominated_idx = np.where(is_pareto)[0][dominated]
            is_pareto[dominated_idx] = False
            is_pareto[i] = True

    return is_pareto

mask = pareto_front_fast(costs)
df_pareto = df_clean[mask].copy()
print(f"Pareto front: {mask.sum()} materials out of {len(df_clean)}")

Pareto front: 555 materials out of 2647


In [17]:
print(cols)
print(len(cols))
print(df_clean.columns.tolist())

['average_voltage', 'capacity_grav', 'energy_grav', 'fracA_charge', 'fracA_discharge', 'stability_charge', 'stability_discharge', 'max_delta_volume']
8
['average_voltage', 'capacity_grav', 'energy_grav', 'fracA_charge', 'fracA_discharge', 'stability_charge', 'stability_discharge', 'max_delta_volume', 'pareto_optimal', 'weighted_score']


In [18]:
from sklearn.preprocessing import MinMaxScaler

# Normalize to [0, 1]
scaler = MinMaxScaler()
df_normalized = pd.DataFrame(
    scaler.fit_transform(df_clean[cols]),
    columns=cols,
    index=df_clean[cols].index
)

# Set your weights (must sum to 1)
weights = {
    "average_voltage":      0.15,
    "capacity_grav":        0.15,
    "energy_grav":          0.10,
    "fracA_charge":         0.10,
    "fracA_discharge":      0.10,
    "stability_charge":     0.15,
    "stability_discharge":  0.10,
    "max_delta_volume":     0.15,
}

# Compute weighted score (flip min objectives)
score = np.zeros(len(df_normalized))
for col, w in weights.items():
    if objectives[col] == "max":
        score += w * df_normalized[col]
    else:
        score += w * (1 - df_normalized[col])  # flip min objectives

df_clean["pareto_optimal"] = mask
df_clean["weighted_score"] = score
df_pareto = df_clean[mask].sort_values("weighted_score", ascending=False)

print(df_pareto.head(20))

      average_voltage  capacity_grav  energy_grav  fracA_charge  \
307          4.370530     313.920945  1372.000778      0.142857   
730          4.613919     245.319178  1131.882694      0.055556   
680          4.685555     245.319178  1149.456506      0.055556   
2257         3.982501     331.270806  1319.286184      0.000000   
1148         4.956461     228.587315  1132.984078      0.090909   
229          2.841711     526.935765  1497.398998      0.000000   
164          3.065887     315.937843   968.629617      0.117647   
36           2.671427     325.701824   870.088561      0.444444   
382          2.254960     450.252852  1015.302187      0.200000   
116          3.401523     276.659709   941.064262      0.222222   
601          3.861625     249.609147   963.896922      0.055556   
1908         5.499463     201.739563  1109.459260      0.083333   
233          5.475817     140.573289   769.753613      0.222222   
194          4.425258     225.569675   998.204014      0.25000

In [19]:
def epsilon_pareto(costs, epsilon=0.05):
    """
    Coarser Pareto front — solutions must be better by at least epsilon.
    Reduces front size dramatically for high-dimensional problems.
    """
    costs = np.array(costs)
    n = len(costs)
    is_pareto = np.ones(n, dtype=bool)

    for i in range(n):
        if is_pareto[i]:
            dominated = (
                np.all(costs[is_pareto] <= costs[i] + epsilon, axis=1) &
                np.any(costs[is_pareto] <  costs[i] - epsilon, axis=1)
            )
            dominated_idx = np.where(is_pareto)[0][dominated]
            is_pareto[dominated_idx] = False
            is_pareto[i] = True

    return is_pareto

# Normalize first so epsilon is meaningful
from sklearn.preprocessing import MinMaxScaler
costs_scaled = MinMaxScaler().fit_transform(costs)

mask_eps = epsilon_pareto(costs_scaled, epsilon=0.05)
print(f"Epsilon-Pareto front: {mask_eps.sum()} materials")

Epsilon-Pareto front: 143 materials


In [20]:
import plotly.express as px

df_plot = df_clean.copy()
df_plot["type"] = np.where(mask, "Pareto Optimal", "Dominated")

fig = px.parallel_coordinates(
    df_plot,
    dimensions=cols,
    color="weighted_score",
    color_continuous_scale=px.colors.sequential.Viridis,
    title="8-Objective Pareto Analysis — Materials Project Electrodes"
)
fig.show()

# Or just the Pareto front
fig2 = px.parallel_coordinates(
    df_pareto,
    dimensions=cols,
    color="weighted_score",
    title="Pareto Front Only"
)
fig2.show()

How Graph Works?
- Each axes represents a property, with one high values at the top and low values at the bottom
- Each line is one materials
- Line color(weighted score) maps brighter/yellow lines to higher scores
-       - Darker/purple lines to lower scores
- Crossings between lines reveals tradeoffs - one material beats at another for a property but loses for another property
- Cluster of parallel lines means similar tradeoffs amon basically interchangeable candidate materials

What to look for:
- lines that stay high for voltage, capacity, energy, etc but low for max delta volume, stability_charge, etc. are great candidates
- If many pareto lines cross heavily between two properties
-        - implies major tradeoff; can't have both

In [25]:
from tabulate import tabulate

def print_ranked_materials(df_clean, df_ids, mask, score, cols, top_n=20):
    # filter rows using boolean mask
    df_out = df_clean[mask].copy()
    df_out["weighted_score"] = score[mask]

    # rejoin ID columns using index
    df_out["battery_id"] = df_ids.loc[df_out.index, "battery_id"]
    df_out["formula_discharge"] = df_ids.loc[df_out.index, "formula_discharge"]

    df_out = df_out.sort_values("weighted_score", ascending=False).reset_index(drop=True)
    df_out.index += 1

    df_display = df_out.head(top_n)[[
        "weighted_score",
        "battery_id",
        "formula_discharge",
        "average_voltage",
        "capacity_grav",
        "energy_grav",
        "fracA_charge",
        "fracA_discharge",
        "stability_charge",
        "stability_discharge",
        "max_delta_volume",
    ]].round(3)

    df_display.columns = [
        "Score",
        "Battery ID",
        "Formula",
        "Voltage",
        "Cap_grav",
        "E_grav",
        "FracA_ch",
        "FracA_dis",
        "Stab_ch",
        "Stab_dis",
        "dVol%",
    ]

    print(f"\nTop {top_n} Pareto-optimal materials:\n")
    print(tabulate(
        df_display,
        headers="keys",
        tablefmt="tsv",
        floatfmt=".3f",
        numalign="right",
        stralign="left",
    ))

# make sure df_ids was defined earlier like this:

print_ranked_materials(df_clean, df_ids, mask, score, cols, top_n=35)

NameError: name 'df_ids' is not defined

In [22]:
print(df_normalized.dtypes)
print(df_normalized.head())

average_voltage        float64
capacity_grav          float64
energy_grav            float64
fracA_charge           float64
fracA_discharge        float64
stability_charge       float64
stability_discharge    float64
max_delta_volume       float64
dtype: object
   average_voltage  capacity_grav  energy_grav  fracA_charge  fracA_discharge  \
0         0.122900       0.224768     0.045221           0.0         0.854651   
1         0.010406       0.633947     0.012687           0.0         0.231728   
2         0.189909       0.361325     0.110196           0.0         0.563953   
3         0.003161       0.161888     0.001055           0.0         0.418605   
4         0.288617       0.390350     0.179939           0.0         0.418605   

   stability_charge  stability_discharge  max_delta_volume  
0          0.099972             0.000000          0.743738  
1          0.024085             0.005332          0.051293  
2          0.898102             0.323399          0.368880  
3      